<div style="background: linear-gradient(135deg, #1e3a5f 0%, #2d5a87 100%); padding: 26px; border-radius: 10px; margin: 30px 0 18px 0;">
<h1 style="color: white; margin: 0; font-size: 2em;">Semana 2 · Práctica</h1>
<p style="color: #a8d4ff; margin: 8px 0 0 0; font-size: 1.05em;">Un agente con búsqueda en los informes, con LangChain, que contesta preguntas solo</p>
</div>
<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Qué vas a hacer</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;">Vas a armar, paso a paso, un agente que contesta preguntas sobre los informes de las cuatro empresas que reportan desde la Argentina: MercadoLibre, YPF, Globant e IRSA.</p>
<ul style="margin: 0 0 12px 0; padding-left: 24px;">
<li style="margin-bottom: 6px;">Usás <strong>LangChain</strong> para las herramientas y el agente (<code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">@tool</code>, <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">create_agent</code>) y para los embeddings.</li>
<li style="margin-bottom: 6px;">Los datos son <strong>20 preguntas nuevas</strong>, con la respuesta esperada y el fragmento donde está, que no vimos en clase.</li>
<li style="margin-bottom: 6px;">La búsqueda la armás vos: primero por vectores, después la medís, después la mejorás.</li>
<li style="margin-bottom: 6px;">Se corrige con el mismo método de la clase: hit-rate@5 para la búsqueda, y un juez para las respuestas.</li>
</ul>
<table style="border-collapse: collapse; margin: 6px 0 14px 0; font-size: 0.95em; width: 100%;">
<thead>
<tr>
<th style="background: #1e3a5f; color: #ffffff; text-align: left; padding: 8px 12px; border: 1px solid #1e3a5f;"></th>
<th style="background: #1e3a5f; color: #ffffff; text-align: left; padding: 8px 12px; border: 1px solid #1e3a5f;"></th>
</tr>
</thead>
<tbody>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><strong>Costo de API</strong></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">cerca de USD 0,20 con <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">gpt-5.4-mini</code>. Estas celdas llaman al modelo en vivo y necesitan tu API key</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><strong>Empresas</strong></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">MELI, YPF, GLOB, IRS: 1.605 fragmentos</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><strong>Autochequeo</strong></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">la última celda dice qué pasos están bien. No hay nada que entregar</td>
</tr>
</tbody>
</table></div></div>

In [ ]:
# Setup: corré esta celda primero.
import json
import os
import re
import subprocess
import sys

sys.path.insert(0, "..")  # la raíz del repo es la raíz de los imports

from typing import Optional

import numpy as np
import pandas as pd
from dotenv import find_dotenv, load_dotenv
from pydantic import BaseModel, Field

#from week_2.lab import corpus, ir_evals, search
#from week_2.lab import tools as lab_tools
from week_2_2.lab import corpus, ir_evals, search
from week_2_2.lab import tools as lab_tools


pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)


USAR_API= True
GEMINI_API_KEY=""
USE_MONGO= False
DESCARGAR_DE_EDGAR= False


load_dotenv(find_dotenv(usecwd=True))

if USAR_API and not(GEMINI_API_KEY or os.getenv("GOOGLE_API_KEY")):
    os.environ["GOOGLE_API_KEY"]= GEMINI_API_KEY
llm.configurar(usar_api= USAR_API, api_key= GEMINI_API_KEY)

cliente= llm.Client()
if USAR_API:
    print("Respuestas: en vivo, desde la API de google gemini")
elif os.getenv("GEMINI_API_KEY"):
    print("Falta la API key: creá un archivo .env con GOOGLE_API_KEY=...")
else:
    print("Respuestas: grabadas(gratis). No hay API key detectada")

EMPRESAS = ("MELI", "YPF", "GLOB", "IRS")
fragmentos = [f for f in corpus.fragmentos() if f["ticker"] in EMPRESAS]
preguntas = corpus.cargar_preguntas("preguntas_tarea")
print(len(fragmentos), "fragmentos |", len(preguntas), "preguntas")
pd.DataFrame(preguntas)[["pregunta", "categoria", "respuesta_esperada"]].head(5)

Directorio actual: c:\Tarea curso Agente IA\week_2_2


'pd.set_option("display.max_colwidth", 120)\npd.set_option("display.width", 200)\n\n\nUSAR_API= True\nGEMINI_API_KEY=""\nUSE_MONGO= False\nDESCARGAR_DE_EDGAR= False\n\n\nload_dotenv(find_dotenv(usecwd=True))\n\nif USAR_API and not(GEMINI_API_KEY or os.getenv("GOOGLE_API_KEY")):\n    os.environ["GOOGLE_API_KEY"]= GEMINI_API_KEY\nllm.configurar(usar_api= USAR_API, api_key= GEMINI_API_KEY)\n\ncliente= llm.Client()\nif USAR_API:\n    print("Respuestas: en vivo, desde la API de google gemini")\nelif os.getenv("GEMINI_API_KEY"):\n    print("Falta la API key: creá un archivo .env con GOOGLE_API_KEY=...")\nelse:\n    print("Respuestas: grabadas(gratis). No hay API key detectada")\n\nEMPRESAS = ("MELI", "YPF", "GLOB", "IRS")\nfragmentos = [f for f in corpus.fragmentos() if f["ticker"] in EMPRESAS]\npreguntas = corpus.cargar_preguntas("preguntas_tarea")\nprint(len(fragmentos), "fragmentos |", len(preguntas), "preguntas")\npd.DataFrame(preguntas)[["pregunta", "categoria", "respuesta_esperada"]].h

<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Qué es lo que vas a usar de LangChain</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><table style="border-collapse: collapse; margin: 6px 0 14px 0; font-size: 0.95em; width: 100%;">
<thead>
<tr>
<th style="background: #1e3a5f; color: #ffffff; text-align: left; padding: 8px 12px; border: 1px solid #1e3a5f;">Pieza</th>
<th style="background: #1e3a5f; color: #ffffff; text-align: left; padding: 8px 12px; border: 1px solid #1e3a5f;">Para qué sirve</th>
<th style="background: #1e3a5f; color: #ffffff; text-align: left; padding: 8px 12px; border: 1px solid #1e3a5f;">En la clase lo hicimos con</th>
</tr>
</thead>
<tbody>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">@tool</code></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">convierte una función de Python en una tool: el nombre, el docstring y los tipos de los parámetros se vuelven la descripción y el schema</td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">el diccionario <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">TOOL_DATO</code> escrito a mano</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">create_agent(modelo, [tools], system_prompt=...)</code></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">arma el loop de tools: llama al modelo, ejecuta lo que pida, vuelve a llamar</td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">loop_de_tools</code> / <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">tools.correr_agente</code></td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">OpenAIEmbeddings</code></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">los vectores</td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">client.embeddings.create</code></td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">ChatOpenAI(...).with_structured_output(Schema)</code></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">una salida con schema, para el juez</td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">client.responses.parse</code></td>
</tr>
</tbody>
</table>
<ul style="margin: 0 0 12px 0; padding-left: 24px;">
<li style="margin-bottom: 6px;">El agente devuelve la lista de mensajes de la conversación: el pedido del usuario, las llamadas a tools (<code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">AIMessage</code> con <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">tool_calls</code>), los resultados (<code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">ToolMessage</code>) y la respuesta final.</li>
<li style="margin-bottom: 6px;">El docstring de la función es la descripción de la tool: LangChain lo manda tal cual. Un docstring de una línea es una descripción vaga, con lo que eso implica.</li>
</ul></div></div>
<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Paso 1: una tool y el agente</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">@tool</code> arriba de una función. El docstring le explica al modelo para qué sirve. Los tipos de los parámetros arman el schema.</p>
<p style="margin: 0 0 12px 0;">Escribí <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">buscar_dato</code> como tool de LangChain, usando por dentro <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">lab_tools.buscar_dato</code> (la de la clase). El docstring tiene que decir qué devuelve, qué campos existen, en qué unidad, y para qué NO sirve. Después armá el agente con <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">create_agent</code> y hacele una pregunta numérica.</p></div></div>

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

CAMPOS = ", ".join(lab_tools.CAMPOS)


@tool
def buscar_dato(ticker: str, campo: str) -> str:
    # TODO 1: el docstring es la descripción de la tool. Escribilo completo.
    '''...'''
    return lab_tools.buscar_dato(ticker, campo)


SYSTEM = ("Sos un analista financiero. Contestás en español, con los datos que te dan las herramientas. "
          "Si las herramientas no alcanzan para contestar, decilo.")

agente_v1 = create_agent("openai:gpt-5.4-mini", [buscar_dato], system_prompt=SYSTEM)

salida = agente_v1.invoke({"messages": [{"role": "user", "content": "¿Cuánto facturó Globant en el último ejercicio?"}]})
for m in salida["messages"]:
    print(type(m).__name__, "|", getattr(m, "tool_calls", "") or "", "|", str(m.content)[:150])

In [ ]:
# ✅ CHECK 1
assert len(buscar_dato.description) > 200, "TODO 1: el docstring tiene que describir campos, unidad y para qué no sirve"
assert any(type(m).__name__ == "ToolMessage" for m in salida["messages"]), "el agente no llamó a la tool"
assert any(x in salida["messages"][-1].content for x in ("2.454", "2,454", "2454", "2.455", "2,455", "2455")), "Globant facturó 2.454,9 millones"
print("OK. El agente llama a la tool y contesta con el dato de la tabla.")

<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Paso 2: vectorizar los fragmentos y buscar</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">OpenAIEmbeddings(model="text-embedding-3-small", dimensions=512)</code> devuelve un vector por texto con <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">.embed_documents([...])</code> y el de una pregunta con <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">.embed_query(...)</code>.</p>
<p style="margin: 0 0 12px 0;">Escribí la clase <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">IndiceVectorial</code>: en el constructor, una matriz con un vector normalizado por fragmento; en <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">buscar</code>, el vector de la pregunta, el producto con la matriz y los <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">k</code> fragmentos de mayor coseno. Vectorizar los 1.605 fragmentos cuesta cerca de USD 0,01.</p></div></div>

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=512)


class IndiceVectorial:
    def __init__(self, fragmentos, texto_de=lambda f: f["text"]):
        self.fragmentos = fragmentos
        # TODO 2a: vectorizá los textos (texto_de(f) para cada fragmento) y guardá la matriz normalizada
        vectores = ...
        self.matriz = ...

    def buscar(self, pregunta, k=5, ticker=None):
        # TODO 2b: el vector de la pregunta, los cosenos con toda la matriz, y los k mejores (con filtro por ticker)
        ...


indice_v1 = IndiceVectorial(fragmentos)
for r in indice_v1.buscar("¿Qué dice YPF sobre Vaca Muerta?", k=3):
    print(f"[{r['id']}] {r['section']} {r['score']:.3f} | {r['text'][:100]!r}")

In [ ]:
# ✅ CHECK 2
assert indice_v1.matriz.shape == (len(fragmentos), 512), "TODO 2a: una fila por fragmento, 512 columnas"
assert np.allclose(np.linalg.norm(indice_v1.matriz, axis=1), 1, atol=1e-3), "TODO 2a: normalizá cada vector"
top = indice_v1.buscar("¿Qué dice YPF sobre Vaca Muerta?", k=3, ticker="YPF")
assert len(top) == 3 and all(r["ticker"] == "YPF" for r in top), "TODO 2b: k resultados, filtrados por ticker"
print("OK. El índice busca por significado.")

<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Paso 3: medir</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">ir_evals.evaluar(buscar, k=5, preguntas=preguntas)</code> corre las 20 preguntas y compara con los fragmentos etiquetados. Corré el eval sobre <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">indice_v1.buscar</code>.</p></div></div>

In [ ]:
# TODO 3: el eval del índice vectorial sobre las 20 preguntas
eval_v1 = ...
print(eval_v1)
eval_v1.por_categoria()

In [ ]:
# ✅ CHECK 3
assert len(eval_v1.df) == 20, "TODO 3: el eval tiene que comparar 20 preguntas"
hit_v1 = eval_v1.hit_rate
print(f"OK. Punto de partida: hit-rate@5 {hit_v1:.0%}, MRR {eval_v1.mrr:.2f}")

<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #4b5563; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;"><span style="font-size: 0.72em; font-weight: 600; letter-spacing: 0.04em; text-transform: uppercase; opacity: 0.85; display: block; margin-bottom: 2px;">👀 Deberías ver</span></div><div style="background: #f3f4f6; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;">Con vectores solos, alrededor del 85 %. Las fallas se concentran en las parafraseadas.</p></div></div>
<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Paso 4: mirar las fallas, y mejorar</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;">Cuatro cosas que vimos en clase, para probar de a una y volver a medir cada vez:</p>
<ol style="margin: 0 0 12px 0; padding-left: 24px;">
<li style="margin-bottom: 6px;"><strong>BM25</strong> para los términos exactos: <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">search.IndiceBM25(fragmentos)</code> ya viene armado.</li>
<li style="margin-bottom: 6px;"><strong>RRF</strong> para combinar las dos listas. Escribila vos: cada lista le suma <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">1 / (60 + posición)</code> a cada fragmento.</li>
<li style="margin-bottom: 6px;"><strong>Un prefijo de contexto</strong> en cada fragmento antes de vectorizar: empresa y sección. <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">IndiceVectorial</code> acepta <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">texto_de=...</code>.</li>
<li style="margin-bottom: 6px;"><strong>Traducir la pregunta al inglés</strong> antes de buscar, con el modelo: los informes están en inglés y BM25 compara palabras. <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">ChatOpenAI(...).with_structured_output(Schema, method="json_schema")</code> devuelve un objeto del schema.</li>
</ol>
<p style="margin: 0 0 12px 0;">Al preparar la práctica, con las tres primeras el índice llegó a 85 %; con la traducción, a 90 %. Anotá en <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">mis_notas</code> qué falló en dos de las preguntas antes de tocar nada.</p>
<p style="margin: 0 0 12px 0;">Una aclaración sobre el umbral: son 20 preguntas, así que cada acierto vale 5 puntos y el intervalo de confianza es ancho. Pasar de 85 % a 95 % son dos preguntas, y con eso no se puede afirmar que tu índice sea mejor. El 90 % está para que hagas el ejercicio completo, no como prueba de nada. El CHECK imprime los intervalos para que los veas.</p></div></div>

In [ ]:
eval_v1.fallas()

In [ ]:
mis_notas = {
    # TODO 4a: (pregunta, en pocas palabras): "qué trajo el índice y por qué no era lo que hacía falta"
}


def rrf(*listas, k=60):
    # TODO 4b: un diccionario id -> puntaje; cada lista suma 1 / (k + posición) empezando en 1; devolver ordenado
    ...


indice_bm25 = search.IndiceBM25(fragmentos)


def prefijo(f):
    # TODO 4c: una línea con la empresa y la sección, por ejemplo "MELI, Item 1A"
    return ...


indice_v2 = IndiceVectorial(fragmentos, texto_de=lambda f: prefijo(f) + "\n" + f["text"])

from langchain_openai import ChatOpenAI


class Traduccion(BaseModel):
    en: str


traductor = ChatOpenAI(model="gpt-5.4-nano").with_structured_output(Traduccion, method="json_schema")


def traducir(pregunta):
    # TODO 4d: pedile al traductor la pregunta en inglés, manteniendo nombres, tickers y cifras
    ...


def buscar_v2(pregunta, k=5, ticker=None):
    en = traducir(pregunta)
    return rrf(indice_v2.buscar(en, 20, ticker), indice_bm25.buscar(en, 20, ticker))[:k]


eval_v2 = ir_evals.evaluar(buscar_v2, k=5, preguntas=preguntas, label="tarea-v2-hibrido")
print(eval_v2)
ir_evals.comparar(eval_v1, eval_v2)

In [ ]:
# ✅ CHECK 4
assert len(mis_notas) >= 2, "TODO 4a: anotá al menos dos fallas"
a = [{"id": "x"}, {"id": "y"}]; b = [{"id": "y"}, {"id": "z"}]
assert [f["id"] for f in rrf(a, b)][0] == "y", "TODO 4b: un fragmento que está en las dos listas tiene que ganar"
assert "Item" in prefijo(fragmentos[0]) and fragmentos[0]["ticker"] in prefijo(fragmentos[0]), "TODO 4c: empresa y sección"
assert "Vaca Muerta" in traducir("¿Qué planes tiene YPF para Vaca Muerta?"), "TODO 4d: la traducción tiene que conservar los nombres"
hit_v2 = eval_v2.hit_rate
print(ir_evals.tabla_con_intervalos(eval_v1, eval_v2))
print(ir_evals.mcnemar(eval_v1, eval_v2))
assert hit_v2 >= 0.9, "Todavía no. Mirá eval_v2.fallas(). Hacen falta 90 % para seguir."
print("\nOK. Podés seguir.")

<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Paso 4-bis: el tamaño de los fragmentos</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;">El corpus viene cortado en 1.200 caracteres con 150 de solapamiento, pero esa es una decisión que se toma midiendo. Es la comparación de la clase, ahora sobre tus cuatro empresas.</p>
<p style="margin: 0 0 12px 0;">A igual presupuesto de contexto (6.000 caracteres) entran 10 fragmentos de 600 o 2 de 2.400. <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">corpus.recortar(tamano)</code> vuelve a cortar los informes y <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">corpus.contesta(traidos, pregunta)</code> compara por posición, así los cortes se pueden comparar entre sí.</p>
<p style="margin: 0 0 12px 0;">Vectorizar los tres cortes cuesta unos 3 centavos.</p></div></div>

In [ ]:
PRESUPUESTO = 6000

def cuantas_contesta(trozos):
    indice = IndiceVectorial(trozos)
    aciertos = 0
    for p in preguntas:
        # TODO 4e: traé fragmentos hasta llenar PRESUPUESTO caracteres, y fijate si contestan
        traidos, total = [], 0
        ...
        aciertos += corpus.contesta(traidos, p)
    return aciertos


def fila(t):
    trozos = [f for f in (fragmentos if t == 1200 else corpus.recortar(t)) if f["ticker"] in EMPRESAS]
    return {"caracteres por fragmento": t, "fragmentos": len(trozos), "entran en 6.000": PRESUPUESTO // t,
            "contesta": f"{cuantas_contesta(trozos)} de 20", "índice (MB)": round(len(trozos) * 512 * 4 / 1e6, 1)}

pd.DataFrame([fila(t) for t in (600, 1200, 2400)]).set_index("caracteres por fragmento")

In [ ]:
# ✅ CHECK 4-bis
assert cuantas_contesta(fragmentos) >= 15, "TODO 4e: con el corpus de 1.200 tendrían que contestar al menos 15"
print("OK. Ya sabés qué tamaño conviene para tus preguntas.")

<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Paso 5: la búsqueda como tool, y el agente con las dos</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;">Escribí <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">buscar_en_informes(query, ticker)</code> como tool sobre <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">buscar_v2</code>, que devuelva JSON con id, sección y texto de cada fragmento (<code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">lab_tools.hacer_buscar_en_informes</code> lo hace, si le pasás un objeto con <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">.buscar</code>). El docstring, completo.</p>
<p style="margin: 0 0 12px 0;">Y el system del agente con las dos reglas de la clase: citar el id entre corchetes, y decir cuando no está.</p></div></div>

In [ ]:
class Buscador:
    buscar = staticmethod(buscar_v2)


@tool
def buscar_en_informes(query: str, ticker: Optional[str] = None) -> str:
    # TODO 5a: el docstring: qué busca, qué devuelve, para qué no sirve, que la consulta va en inglés, qué tickers hay
    '''...'''
    return lab_tools.hacer_buscar_en_informes(Buscador(), k=5)(query, ticker)


# TODO 5b: dos reglas más: citar el id del fragmento entre corchetes, y decir cuando no está en los informes
SYSTEM_CITA = SYSTEM + " ..."

agente_v2 = create_agent("openai:gpt-5.4-mini", [buscar_dato, buscar_en_informes], system_prompt=SYSTEM_CITA)

salida = agente_v2.invoke({"messages": [{"role": "user", "content": preguntas[0]["pregunta"]}]})
print(salida["messages"][-1].content)

In [ ]:
# ✅ CHECK 5
assert len(buscar_en_informes.description) > 200, "TODO 5a: el docstring completo"
assert "[" in SYSTEM_CITA and "no está" in SYSTEM_CITA.lower(), "TODO 5b: las dos reglas"
assert re.search(r"\[[A-Z]+-\d{4}\]", salida["messages"][-1].content), "la respuesta tiene que citar un id como [MELI-0214]"
print("OK. El agente busca en los informes y cita.")

<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Paso 6: medir las respuestas con un juez</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;">Las 20 preguntas tienen una respuesta esperada. Un juez binario compara: ¿la respuesta del agente contiene la información esperada? Sí o no.</p>
<p style="margin: 0 0 12px 0;">Armá el juez con <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">ChatOpenAI(model="gpt-5.4-mini").with_structured_output(Correccion, method="json_schema")</code> y corré las 20 preguntas por el agente. Cuesta unos 5 centavos.</p></div></div>

In [ ]:
from langchain_openai import ChatOpenAI


class Correccion(BaseModel):
    correcta: bool = Field(description="True si la respuesta contiene la información de la respuesta esperada, aunque agregue más")
    motivo: str


# TODO 6a: el juez, con salida estructurada
juez = ...


def juzgar(pregunta, esperada, respuesta):
    # TODO 6b: un mensaje con la pregunta, la respuesta esperada y la respuesta del agente; devolver el objeto Correccion
    ...


filas = []
for p in preguntas:
    r = agente_v2.invoke({"messages": [{"role": "user", "content": p["pregunta"]}]})["messages"][-1].content
    v = juzgar(p["pregunta"], p["respuesta_esperada"], r)
    filas.append({"pregunta": p["pregunta"], "correcta": v.correcta, "cita": bool(re.search(r"\[[A-Z]+-\d{4}\]", r)),
                  "respuesta": r[:160], "motivo": v.motivo})
resultados = pd.DataFrame(filas)
print(f"correctas: {int(resultados['correcta'].sum())} de 20 | con cita: {int(resultados['cita'].sum())} de 20")
resultados[~resultados["correcta"]][["pregunta", "respuesta", "motivo"]]

In [ ]:
# ✅ CHECK 6
assert isinstance(juzgar("¿Cuánto facturó Globant?", "2.454 millones de USD", "Globant facturó USD 2.454,9 millones [GLOB-0001]"), Correccion), "TODO 6: juzgar devuelve un Correccion"
correctas = int(resultados["correcta"].sum())
assert correctas >= 15, "Todavía no. Mirá las que no: ¿falló la búsqueda o la respuesta? Hacen falta 15 de 20."
print(f"OK. {correctas} de 20 correctas.")

<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Paso 7: que corra solo</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;">Un sistema real no contesta una pregunta en un notebook: recibe una lista de preguntas y las contesta sin que nadie mire.</p>
<ol style="margin: 0 0 12px 0; padding-left: 24px;">
<li style="margin-bottom: 6px;">Abrí <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">week_2/produccion/responder.py</code>. Ya tiene los imports, las rutas y el <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">main</code>.</li>
<li style="margin-bottom: 6px;">Pegá tu índice, tus dos tools y tu system en los lugares marcados.</li>
<li style="margin-bottom: 6px;">Escribí <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">correr</code>: lee las preguntas, saltea las que ya están contestadas en <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">artifacts/respuestas.jsonl</code>, contesta el resto, anota los errores sin frenar, y deja <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">artifacts/responder_reporte.json</code>.</li>
<li style="margin-bottom: 6px;">Probalo desde <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">week_2/</code>:</li>
</ol>
<pre style="background: #f1f5f9; color: #0f172a; border: 1px solid #cbd5e1; padding: 14px 16px; border-radius: 8px; overflow-x: auto; font-size: 0.88em; line-height: 1.45; margin: 6px 0 14px 0;"><code style="background: transparent; color: #0f172a; padding: 0;">python produccion/responder.py --reset           # contesta las preguntas de produccion/preguntas.jsonl
python produccion/responder.py                   # segunda corrida: no tiene que contestar nada
python produccion/check_tarea.py                 # los tres controles
</code></pre>
<p style="margin: 0 0 12px 0;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">check_tarea.py</code> hace tres controles: 15 de 20 correctas y 15 de 20 con cita, que la segunda corrida no repita trabajo, y que una pregunta vacía no frene a las demás.</p>
<p style="margin: 0 0 12px 0;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">produccion/responder_solucion.py</code> es una solución posible. Conviene mirarla después de intentarlo.</p></div></div>

In [ ]:
# Dejalo vacío para revisar tu produccion/responder.py
ARGUMENTOS = []

In [ ]:
# ✅ CHECK 7: corre los tres controles. Tarda unos 4 minutos y cuesta cerca de USD 0,10.
controles = subprocess.run([sys.executable, "produccion/check_tarea.py", *ARGUMENTOS], capture_output=True, text=True)
print(controles.stdout[-2500:] or controles.stderr[-1500:])
assert controles.returncode == 0, "Todavía no pasa los tres controles. Arriba dice cuál falta."
print("OK.")

<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Paso 8 (opcional): achicar el índice</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;">Con 1.605 fragmentos la matriz entra en cualquier lado. Con un millón, no. Probá la versión binaria sobre tu propia matriz, sin llamar a la API: un bit por dimensión en lugar de cuatro bytes.</p>
<p style="margin: 0 0 12px 0;">Completá el rescate: los bits sirven para traer 50 candidatos rápido, y a esos 50 se les calcula el coseno de verdad.</p>
<p style="margin: 0 0 12px 0;">En el corpus de la clase, con 3.356 fragmentos, la versión binaria sola pierde 10 puntos y el rescate los recupera. En el tuyo, con la mitad de fragmentos, puede no perder nada, y el rescate hasta puede perder alguna: reordena 50 candidatos con el coseno real, así que un fragmento que los bits ponían tercero puede terminar séptimo. Con 20 preguntas eso es un acierto de diferencia, que ya sabemos que no se mide. Lo que no se discute es el tamaño.</p></div></div>

In [ ]:
M = indice_v2.matriz
# la misma consulta que usa tu buscador: traducida al inglés
vector_de = {p["pregunta"]: np.array(embeddings.embed_query(traducir(p["pregunta"])), dtype="float32") for p in preguntas}
bits = np.packbits(M > 0, axis=1)

def hit_rate(score_de):
    return sum(any(indice_v2.fragmentos[i]["id"] in p["relevantes"]
                   for i in np.argsort(-score_de(p["pregunta"]))[:5]) for p in preguntas) / len(preguntas)

def score_binario(pregunta, rescatar=0):
    q = vector_de[pregunta]
    distancia = np.unpackbits(bits ^ np.packbits(q > 0), axis=1).sum(axis=1)
    if not rescatar:
        return -distancia.astype("float32")
    # TODO 8: los `rescatar` mejores según los bits, reordenados con el coseno real
    ...

print(f"float32  {M.nbytes / 1e6:.1f} MB  hit-rate {hit_rate(lambda q: M @ vector_de[q]):.0%}")
print(f"binario  {bits.nbytes / 1e6:.1f} MB  hit-rate {hit_rate(lambda q: score_binario(q)):.0%}")
print(f"+rescate {bits.nbytes / 1e6:.1f} MB  hit-rate {hit_rate(lambda q: score_binario(q, 50)):.0%}")

In [ ]:
# ✅ CHECK 8 (opcional)
una = preguntas[0]["pregunta"]
puntuados = int((score_binario(una, 50) > -1e8).sum())
assert puntuados == 50, f"TODO 8: el rescate tiene que puntuar los 50 mejores según los bits, no {puntuados}"
mejores_por_bits = np.argsort(np.unpackbits(bits ^ np.packbits(vector_de[una] > 0), axis=1).sum(axis=1))[:50]
assert np.argsort(-score_binario(una, 50))[0] in mejores_por_bits, "TODO 8: el primero del rescate tiene que salir de esos 50"
print(f"OK. El índice pasa de {M.nbytes / 1e6:.1f} MB a {bits.nbytes / 1e6:.1f} MB.")

<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Autochequeo</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;">La celda de abajo revisa los siete pasos y dice cuáles están bien y cuáles faltan. No hay nada que entregar.</p></div></div>

In [ ]:
# ✅ Autochequeo: no llama al modelo, mira lo que ya corriste
g = globals()
pasos = [
    ("1. La tool y el agente", "salida" in g and len(getattr(g.get("buscar_dato"), "description", "")) > 200),
    ("2. El índice vectorial", "indice_v1" in g and g["indice_v1"].matriz.shape[1] == 512),
    ("3. Medir", "eval_v1" in g and len(g["eval_v1"].df) == 20),
    ("4. Híbrido con 90 % o más", g.get("hit_v2", 0) >= 0.9),
    ("5. La búsqueda como tool, con cita", "agente_v2" in g and "[" in g.get("SYSTEM_CITA", "")),
    ("6. Juez: 15 de 20 o más", "resultados" in g and int(g["resultados"]["correcta"].sum()) >= 15),
    ("7. El script pasa los tres controles", getattr(g.get("controles"), "returncode", 1) == 0),
    ("4-bis. El tamaño de fragmento, medido", "cuantas_contesta" in g),
    ("8 (opcional). El índice achicado", "score_binario" in g),
]
for nombre, ok in pasos:
    print("✅" if ok else "⬜", nombre)

obligatorios = pasos[:7]
hechos_obl = sum(ok for _, ok in obligatorios)
print(f"\n{hechos_obl} de {len(obligatorios)} pasos completos." + (" Terminaste la práctica." if hechos_obl == len(obligatorios) else ""))

<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Para terminar: cómo se mantiene en el tiempo</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><ul style="margin: 0 0 12px 0; padding-left: 24px;">
<li style="margin-bottom: 6px;">Cuando llega un informe nuevo se cortan sus fragmentos y se agregan al índice. No hace falta rehacer todo, porque los vectores de los fragmentos viejos no cambian. Con una base vectorial (Chroma, pgvector, Qdrant) es <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">collection.add(...)</code>.</li>
<li style="margin-bottom: 6px;">Guardá las preguntas que hace la gente y cada tanto etiquetá veinte con el fragmento correcto. Ese set vale más que el que escribimos nosotros.</li>
<li style="margin-bottom: 6px;"><strong>La VM y el cron</strong> son los mismos de la semana 1. La diferencia es que este script lee una cola de preguntas; en un sistema real la cola es una tabla o un servicio, y el script corre cada pocos minutos.</li>
<li style="margin-bottom: 6px;">Si no todos pueden ver todos los documentos, el filtro va en la búsqueda (<code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">ticker=</code> acá; el usuario y su grupo en producción), nunca en el prompt.</li>
</ul></div></div>
<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Recursos</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><table style="border-collapse: collapse; margin: 6px 0 14px 0; font-size: 0.95em; width: 100%;">
<thead>
<tr>
<th style="background: #1e3a5f; color: #ffffff; text-align: left; padding: 8px 12px; border: 1px solid #1e3a5f;">Tema</th>
<th style="background: #1e3a5f; color: #ffffff; text-align: left; padding: 8px 12px; border: 1px solid #1e3a5f;">Sitio</th>
</tr>
</thead>
<tbody>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://docs.langchain.com/oss/python/langchain/tools">Tools en LangChain</a></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">docs.langchain.com</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://docs.langchain.com/oss/python/langchain/agents">Agentes: <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">create_agent</code></a></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">docs.langchain.com</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://docs.langchain.com/oss/python/langchain/middleware">Middleware: reintentos, límites de tools, resumen del contexto</a></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">docs.langchain.com</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://developers.openai.com/api/docs/guides/function-calling">Function calling en la Responses API de OpenAI</a></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">developers.openai.com</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://developers.openai.com/api/docs/guides/embeddings">Embeddings de OpenAI</a></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">developers.openai.com</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://developers.openai.com/api/docs/guides/tools"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">file_search</code>: el retrieval administrado de OpenAI</a></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">developers.openai.com</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://www.anthropic.com/engineering/contextual-retrieval">Contextual retrieval, Anthropic</a></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">anthropic.com</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://www.trychroma.com/research/evaluating-chunking">Evaluar chunking, Chroma</a></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">trychroma.com</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://www.trychroma.com/research/context-rot">Context rot, Chroma</a></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">trychroma.com</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://platform.claude.com/cookbook/capabilities-retrieval-augmented-generation-guide">Guía de RAG con evaluación, Anthropic Cookbook</a></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">platform.claude.com</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://hamel.dev/blog/posts/llm-judge/">Cómo armar un juez con el modelo, Hamel Husain</a></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">hamel.dev</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://567-labs.github.io/systematically-improving-rag/">Systematically improving RAG, Jason Liu</a></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">567-labs.github.io</td>
</tr>
</tbody>
</table>
<p style="margin: 0 0 12px 0;">Versiones con las que se probó esta práctica: <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">langchain 1.4.2</code>, <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">langchain-openai 1.6.3</code>. Están en <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">requirements-w2.txt</code>.</p></div></div>